# Social Media Sentiment Analysis

End-to-end NLP workflow on a social media posts dataset: data cleaning, exploratory data analysis, text preprocessing/tokenization, and visualization of sentiment and engagement patterns.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


## 2. Load Data

Dataset is read from the local `data/` folder (no Google Drive dependency).

In [ ]:
data = pd.read_csv("../data/sentimentdataset.csv")
data.head()


## 3. Data Cleaning

Drop redundant index columns carried over from the raw export, then check structure, duplicates, and missing values.

In [ ]:
data = data.drop(columns=["Unnamed: 0.1", "Unnamed: 0"])


In [ ]:
data.info()


In [ ]:
data.shape


In [ ]:
data.describe()


In [ ]:
data.describe(include="O")


In [ ]:
data["Platform"].value_counts()


In [ ]:
data.duplicated().sum()


In [ ]:
data = data.drop_duplicates()
data.duplicated().sum()


In [ ]:
data.isnull().sum()


In [ ]:
data.dtypes


## 4. Target Variable: Sentiment

In [ ]:
data["Sentiment"].value_counts()


In [ ]:
sentiments = data["Sentiment"].unique()
print(sentiments)
print("Number of unique sentiment labels:", data["Sentiment"].nunique())


## 5. Preprocessing: Encoding and Train/Test Split

In [ ]:
X = data["Text"]
y = data["Sentiment"]


In [ ]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
dict(zip(encoder.classes_, range(len(encoder.classes_))))


In [ ]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)
print(train_data.shape)
print(test_data.shape)


## 6. Tokenization

Using Keras' `Tokenizer` to build a vocabulary from the training text and pad sequences to a fixed length for downstream modeling.

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["Text"])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["Text"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["Text"]), maxlen=200)


In [ ]:
X_train


In [ ]:
X_test


In [ ]:
Y_train = train_data["Sentiment"]
Y_test = test_data["Sentiment"]
Y_train.head()


## 7. Exploratory Data Analysis & Visualization

### Top 20 Sentiment Categories

In [ ]:
top_sentiments = data["Sentiment"].value_counts().head(20)

plt.figure(figsize=(12, 6))
top_sentiments.plot(kind="bar")
plt.title("Top 20 Sentiments")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../images/top_20_sentiments.png", dpi=150)
plt.show()


In [ ]:
data["Sentiment"].value_counts().tail(20)


### Numerical Feature Ranges

In [ ]:
numerical_columns = data[["Day", "Month", "Year", "Likes", "Retweets"]]

for col in numerical_columns.columns:
    print(f"Minimum {col}: {data[col].min()} | Maximum {col}: {data[col].max()}")


### Platform Engagement: Top Platforms by Total Likes

In [ ]:
top_likes_platform = data.groupby("Platform")["Likes"].sum().nlargest(10)

plt.figure(figsize=(10, 6))
top_likes_platform.plot(kind="bar")
plt.title("Top Platforms by Total Likes")
plt.xlabel("Platform")
plt.ylabel("Total Likes")
plt.tight_layout()
plt.savefig("../images/top_platforms_by_likes.png", dpi=150)
plt.show()


## 8. Summary

- Cleaned the raw export by removing duplicate index columns, dropping duplicate rows, and confirming there were no missing values.
- Explored the distribution of the `Sentiment` label (279 unique fine-grained sentiment tags) along with platform and engagement statistics.
- Encoded sentiment labels with `LabelEncoder` and split the data into train/test sets (80/20).
- Tokenized post text with a Keras `Tokenizer` (5,000-word vocabulary) and padded sequences to length 200, producing model-ready `X_train`/`X_test` arrays.
- Next steps: train a classification model (e.g. LSTM/GRU or a transformer-based encoder) on the tokenized sequences and evaluate against `Y_test`.
